In [1]:
!pip install pandarallel

  Preparing metadata (setup.py) ... done
  Created wheel for pandarallel: filename=pandarallel-1.6.5-py3-none-any.whl size=16674 sha256=ed69a55646e17f4ebabb140132b23316465f3577dbdaacda70bbe521d5c35cdb
  Stored in directory: /root/.cache/pip/wheels/46/f9/0d/40c9cd74a7cb8dc8fe57e8d6c3c19e2c730449c0d3f2bf66b5
Successfully built pandarallel


In [2]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 22.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=6491e8fc316828573d645b351b2b24c5094a2e57b4442b0c89972570f6aeb527
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
import re
import pandas as pd
import spacy
from langdetect import detect
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from pandarallel import pandarallel

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [6]:
data = pd.read_csv('2024-04-22.csv')
data.head()

,id,doi,author,title,publication_year,journal,issn_l,first_page,last_page,volume,...,concepts,tags_level,tags_score,retrieved,source,journal_issn,field,abstract_length,umap_v1,umap_v2
0,https://openalex.org/W3196334260,https://doi.org/10.15452/studiagermanistica.20...,Eva Polášková,Zum Sprachstil in Wissensvermittlungsfernsehse...,2021,Acta Facultatis Philosophicae Universitatis Os...,1803-408X,105,132,NaN,...,Czech; German; Task (project management); Qual...,2; 2; 2; 2; 2; 1; 1; 1; 0; 0; 0; 0; 0; 1; 1; 0...,0.8670031; 0.7713295; 0.5688958; 0.4573457; 0....,2023-04-30,openalex,ACTA FACULTATIS PHILOSOPHICAE UNIVERSITATIS OS...,linguistics,889,-0.512336,-0.478651
1,https://openalex.org/W3196416014,https://doi.org/10.15452/studiagermanistica.20...,Annegret Lidzba; Krystian Suchorab,Phraseologismen im Bereich der Sexualität. Ein...,2021,Acta Facultatis Philosophicae Universitatis Os...,1803-408X,15,28,NaN,...,Taboo; German; Linguistics; Czech; Typology; S...,2; 2; 1; 2; 2; 0; 2; 2; 0; 0; 1; 1; 0; 1,0.91238606; 0.81760263; 0.5735172; 0.499166; 0...,2023-04-30,openalex,ACTA FACULTATIS PHILOSOPHICAE UNIVERSITATIS OS...,linguistics,1260,0.005452,-0.437668
2,https://openalex.org/W3197299675,https://doi.org/10.15452/studiagermanistica.20...,Radim Maňák,Tendenz zum präpositionalen Genitiv am Beispie...,2021,Acta Facultatis Philosophicae Universitatis Os...,1803-408X,43,50,NaN,...,Genitive case; Linguistics; Mathematics; Human...,3; 1; 0; 1; 0; 2,0.9517493; 0.5148846; 0.35445818; 0.3218974; 0...,2023-04-30,openalex,ACTA FACULTATIS PHILOSOPHICAE UNIVERSITATIS OS...,linguistics,491,0.301636,-0.595104
3,https://openalex.org/W3197696444,https://doi.org/10.15452/studiagermanistica.20...,Beáta Szép,Deutsch als Ausgangssprache für die Erneuerung...,2021,Acta Facultatis Philosophicae Universitatis Os...,1803-408X,89,96,NaN,...,Meaning (existential); Terminology; Vocabulary...,2; 2; 2; 0; 1; 0; 1,0.7451532; 0.72004; 0.4745477; 0.41225076; 0.3...,2023-04-30,openalex,ACTA FACULTATIS PHILOSOPHICAE UNIVERSITATIS OS...,linguistics,519,-0.048791,-0.507939
4,https://openalex.org/W3197861932,https://doi.org/10.15452/studiagermanistica.20...,Sonila Sadikaj,Die Kuh vom Eis bringen. Landwirtschaft als me...,2021,Acta Facultatis Philosophicae Universitatis Os...,1803-408X,51,72,NaN,...,German; Problem of universals; Linguistics; Po...,2; 2; 1; 0; 0,0.6929071; 0.63309395; 0.61870193; 0.39422876;...,2023-04-30,openalex,ACTA FACULTATIS PHILOSOPHICAE UNIVERSITATIS OS...,linguistics,748,0.149425,-0.595887


In [7]:
data2 = pd.read_csv('annotated.csv')

data = pd.merge(data, data2[['id', 'type']], on='id', how='left')


data.head()

,id,doi,author,title,publication_year,journal,issn_l,first_page,last_page,volume,...,tags_level,tags_score,retrieved,source,journal_issn,field,abstract_length,umap_v1,umap_v2,type
0,https://openalex.org/W3196334260,https://doi.org/10.15452/studiagermanistica.20...,Eva Polášková,Zum Sprachstil in Wissensvermittlungsfernsehse...,2021,Acta Facultatis Philosophicae Universitatis Os...,1803-408X,105,132,NaN,...,2; 2; 2; 2; 2; 1; 1; 1; 0; 0; 0; 0; 0; 1; 1; 0...,0.8670031; 0.7713295; 0.5688958; 0.4573457; 0....,2023-04-30,openalex,ACTA FACULTATIS PHILOSOPHICAE UNIVERSITATIS OS...,linguistics,889,-0.512336,-0.478651,language teaching
1,https://openalex.org/W3196416014,https://doi.org/10.15452/studiagermanistica.20...,Annegret Lidzba; Krystian Suchorab,Phraseologismen im Bereich der Sexualität. Ein...,2021,Acta Facultatis Philosophicae Universitatis Os...,1803-408X,15,28,NaN,...,2; 2; 1; 2; 2; 0; 2; 2; 0; 0; 1; 1; 0; 1,0.91238606; 0.81760263; 0.5735172; 0.499166; 0...,2023-04-30,openalex,ACTA FACULTATIS PHILOSOPHICAE UNIVERSITATIS OS...,linguistics,1260,0.005452,-0.437668,lexicography
2,https://openalex.org/W3197299675,https://doi.org/10.15452/studiagermanistica.20...,Radim Maňák,Tendenz zum präpositionalen Genitiv am Beispie...,2021,Acta Facultatis Philosophicae Universitatis Os...,1803-408X,43,50,NaN,...,3; 1; 0; 1; 0; 2,0.9517493; 0.5148846; 0.35445818; 0.3218974; 0...,2023-04-30,openalex,ACTA FACULTATIS PHILOSOPHICAE UNIVERSITATIS OS...,linguistics,491,0.301636,-0.595104,typology
3,https://openalex.org/W3197696444,https://doi.org/10.15452/studiagermanistica.20...,Beáta Szép,Deutsch als Ausgangssprache für die Erneuerung...,2021,Acta Facultatis Philosophicae Universitatis Os...,1803-408X,89,96,NaN,...,2; 2; 2; 0; 1; 0; 1,0.7451532; 0.72004; 0.4745477; 0.41225076; 0.3...,2023-04-30,openalex,ACTA FACULTATIS PHILOSOPHICAE UNIVERSITATIS OS...,linguistics,519,-0.048791,-0.507939,onomastics
4,https://openalex.org/W3197861932,https://doi.org/10.15452/studiagermanistica.20...,Sonila Sadikaj,Die Kuh vom Eis bringen. Landwirtschaft als me...,2021,Acta Facultatis Philosophicae Universitatis Os...,1803-408X,51,72,NaN,...,2; 2; 1; 0; 0,0.6929071; 0.63309395; 0.61870193; 0.39422876;...,2023-04-30,openalex,ACTA FACULTATIS PHILOSOPHICAE UNIVERSITATIS OS...,linguistics,748,0.149425,-0.595887,language_discription


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82965 entries, 0 to 82964
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                82965 non-null  object 
 1   doi               78246 non-null  object 
 2   author            82965 non-null  object 
 3   title             82874 non-null  object 
 4   publication_year  82965 non-null  int64  
 5   journal           82527 non-null  object 
 6   issn_l            82388 non-null  object 
 7   first_page        61915 non-null  object 
 8   last_page         61685 non-null  object 
 9   volume            64864 non-null  object 
 10  issue             61960 non-null  object 
 11  is_retracted      82965 non-null  bool   
 12  cited_by_count    82965 non-null  int64  
 13  abstract          82965 non-null  object 
 14  concepts          82962 non-null  object 
 15  tags_level        82962 non-null  object 
 16  tags_score        82962 non-null  object

In [9]:
data_cleaned = data.copy()

columns_to_drop = ['first_page', 'last_page', 'volume', 'issue', 'is_retracted', 'tags_level', 'tags_score', 'retrieved',	'source', 'umap_v1', 'umap_v2']
data_cleaned = data_cleaned.drop(columns=columns_to_drop)

data_cleaned.head()

,id,doi,author,title,publication_year,journal,issn_l,cited_by_count,abstract,concepts,journal_issn,field,abstract_length,type
0,https://openalex.org/W3196334260,https://doi.org/10.15452/studiagermanistica.20...,Eva Polášková,Zum Sprachstil in Wissensvermittlungsfernsehse...,2021,Acta Facultatis Philosophicae Universitatis Os...,1803-408X,0,School is not the only institution that educat...,Czech; German; Task (project management); Qual...,ACTA FACULTATIS PHILOSOPHICAE UNIVERSITATIS OS...,linguistics,889,language teaching
1,https://openalex.org/W3196416014,https://doi.org/10.15452/studiagermanistica.20...,Annegret Lidzba; Krystian Suchorab,Phraseologismen im Bereich der Sexualität. Ein...,2021,Acta Facultatis Philosophicae Universitatis Os...,1803-408X,0,"People’s sex life is very often, if not always...",Taboo; German; Linguistics; Czech; Typology; S...,ACTA FACULTATIS PHILOSOPHICAE UNIVERSITATIS OS...,linguistics,1260,lexicography
2,https://openalex.org/W3197299675,https://doi.org/10.15452/studiagermanistica.20...,Radim Maňák,Tendenz zum präpositionalen Genitiv am Beispie...,2021,Acta Facultatis Philosophicae Universitatis Os...,1803-408X,0,This article focuses on the use of the preposi...,Genitive case; Linguistics; Mathematics; Human...,ACTA FACULTATIS PHILOSOPHICAE UNIVERSITATIS OS...,linguistics,491,typology
3,https://openalex.org/W3197696444,https://doi.org/10.15452/studiagermanistica.20...,Beáta Szép,Deutsch als Ausgangssprache für die Erneuerung...,2021,Acta Facultatis Philosophicae Universitatis Os...,1803-408X,0,The study presented in this paper reviews the ...,Meaning (existential); Terminology; Vocabulary...,ACTA FACULTATIS PHILOSOPHICAE UNIVERSITATIS OS...,linguistics,519,onomastics
4,https://openalex.org/W3197861932,https://doi.org/10.15452/studiagermanistica.20...,Sonila Sadikaj,Die Kuh vom Eis bringen. Landwirtschaft als me...,2021,Acta Facultatis Philosophicae Universitatis Os...,1803-408X,0,The aim of this contrastive study is to examin...,German; Problem of universals; Linguistics; Po...,ACTA FACULTATIS PHILOSOPHICAE UNIVERSITATIS OS...,linguistics,748,language_discription


In [10]:
duplicates = data_cleaned[data_cleaned.duplicated(subset='id')]
duplicates.shape[0]

3426

In [11]:
empty_rows = data_cleaned[data_cleaned[['title', 'abstract']].isnull().any(axis=1)]
empty_rows.shape[0]

91

In [12]:
data_cleaned = data_cleaned.drop_duplicates(subset='id')
data_cleaned = data_cleaned.dropna(subset=['title', 'abstract'])

In [13]:
data_cleaned.to_csv('abstracts_cleaned.csv', index=False)

In [22]:
# init parallel processing
pandarallel.initialize(progress_bar=False)

additional_stop_words = [
    'study', 'research', 'method', 'result', 'findings', 'paper', 'theory', 'data',
    'approach', 'analysis', 'review', 'article'
]

# main class for text preprocessing
class TextPreprocessor:
    def __init__(self):
        # get regex filters
        self.html_regex = re.compile(r'<[^>]*>')
        self.url_regex = re.compile(r'https?://\S+|www\.\S+')
        self.spacelike_regex = re.compile(r'\s+')

        # initialise spaCy for English
        self.nlp = spacy.load("en_core_web_sm")

        # stopwords
        self.stop_words = set(stopwords.words('english'))
        self.stop_words.update(additional_stop_words)

    def _clean_noise(self, text: str) -> str:
        """
        Clean html-tags, url-links, multiple space-like chars
        :param text: text to be cleaned
        :return: preprocessed text
        """
        text = self.html_regex.sub(' ', text)
        text = self.url_regex.sub(' ', text)
        text = self.spacelike_regex.sub(' ', text)
        return text.strip().lower()

    def process_string(self, text: str) -> str:
        if not isinstance(text, str) or not text:
            return ""

        cleaned = self._clean_noise(text)
        if not cleaned:
            return ""

        # spaCy lemmatization
        doc = self.nlp(cleaned)
        lemmatised_text = []

        for token in doc:
            lemma = token.lemma_.lower().strip()

            # skip spaces, punctuation and empty tokens
            if token.is_space or token.is_punct:
                continue

            # skip stopwords
            if lemma in self.stop_words:
                continue

            # skip empty / very short / non-alphabetic tokens
            if not lemma or len(lemma) < 1:
                continue
            if not any(char.isalpha() for char in lemma):
                continue

            lemmatised_text.append(lemma)

        return " ".join(lemmatised_text)

    def detect_language(self, text: str) -> str:
        """ Detect language of the input text """
        try:
            return detect(text)
        except:
            return 'unknown'

    def preprocessing_pipeline(self, input_df: pd.DataFrame) -> pd.DataFrame:
        input_df = input_df.copy()

        # Detect language of title and abstract
        input_df['title_lang'] = input_df['title'].apply(self.detect_language)
        input_df['abstract_lang'] = input_df['abstract'].apply(self.detect_language)

        # add column for cleaned and lemmatised title and abstract
        input_df['title_lemmatised'] = input_df['title'].parallel_apply(self.process_string)
        input_df['abstract_lemmatised'] = input_df['abstract'].parallel_apply(self.process_string)

        # create index text based on language
        input_df['index_text_lemmatised'] = input_df.apply(
            lambda row: (
                row['title_lemmatised'] + ' ' + row['abstract_lemmatised']
                if row['title_lang'] == 'en' else row['abstract_lemmatised']
            ),
            axis=1
        )

        # clean empty title and abstract
        input_df = input_df[
            ~(
                (input_df['title_lemmatised'] == '') &
                (input_df['abstract_lemmatised'] == '')
            )
        ]

        # Drop language-related columns since they're no longer needed
        input_df = input_df.drop(columns=['title_lang', 'abstract_lang'])

        return input_df.reset_index(drop=True)

INFO: Pandarallel will run on 1 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [23]:
text_preprocessor = TextPreprocessor()
processed_data = text_preprocessor.preprocessing_pipeline(data_cleaned)

In [24]:
processed_data.to_csv('abstracts_lemmatized.csv', index=False)
print(processed_data[['id', 'title_lemmatised', 'abstract_lemmatised', 'index_text_lemmatised']].sample(5))

                                     id  \
28826  https://openalex.org/W1992901257   
69870  https://openalex.org/W3201632722   
75827  https://openalex.org/W1975056813   
43953  https://openalex.org/W2040624358   
25082  https://openalex.org/W2067740957   

                                        title_lemmatised  \
28826  commentary filler syllable status emerge gramm...   
69870  dveh verzijah romana poti razlike prikrite v v...   
75827                    dynamic ter malay bahasa melayu   
43953                    robert lowth strong verb system   
25082  vertical horizontal ethnography language polic...   

                                     abstract_lemmatised  \
28826  peters set useful initial structure classify e...   
69870  izidor cankar publish novel poti twice existen...   
75827  undertake fine grain semantic multiple us poly...   
43953  set trace origin grammatical rule strong verb ...   
25082  discuss ethnography bilingual intercultural ed...   

                   

Test

In [16]:
import time

sample_size = 100
sampled_data = data_cleaned.sample(n=sample_size, random_state=42)

print(sampled_data[['id', 'title', 'abstract']].head())

                                     id  \
67935  https://openalex.org/W2748365223   
77894  https://openalex.org/W1969459782   
36319  https://openalex.org/W2990383546   
6654   https://openalex.org/W2076067761   
12873  https://openalex.org/W2270030679   

                                                   title  \
67935  A corpus-based analysis of the verb pleróo in ...   
77894  Some Aspects of Students’ Viewpoints on Foreig...   
36319      Glottal wave forms for normal female speakers   
6654   Mexican Students’ Identities in Their Language...   
12873  Acoustic Analysis of Voice-Onset Time in Taiwa...   

                                                abstract  
67935  This is a corpus-based study of the developmen...  
77894  The article deals with future foreign language...  
36319  Glottal air flow and subglottal air pressure a...  
6654   The growth of the Mexican population in the U....  
12873  The purpose of this study is to present a thor...  


In [13]:
# 2
text_preprocessor = TextPreprocessor()
start_time = time.time()

processed_sampled_data = text_preprocessor.preprocessing_pipeline(sampled_data)
end_time = time.time()

print(f"Время предобработки для {sample_size} строк: {end_time - start_time:.2f} секунд")
print(processed_sampled_data[['id', 'title_lemmatised', 'abstract_lemmatised', 'index_text_lemmatised']].head())

Время предобработки для 100 строк: 8.87 секунд
                                 id  \
0  https://openalex.org/W2748365223   
1  https://openalex.org/W1969459782   
2  https://openalex.org/W2990383546   
3  https://openalex.org/W2076067761   
4  https://openalex.org/W2270030679   

                                    title_lemmatised  \
0              corpus base verb pleróo ancient greek   
1  aspect student viewpoint foreign language teac...   
2            glottal wave form normal female speaker   
3  mexican student identity language use u.s high...   
4  acoustic voice onset time taiwan mandarin japa...   

                                 abstract_lemmatised  \
0  corpus base development verb pleróo ancient gr...   
1  deal future foreign language teacher viewpoint...   
2  glottal air flow subglottal air pressure relat...   
3  growth mexican population u.s midwest also ref...   
4  purpose present thorough acoustic investigatio...   

                               index_text_le

In [14]:
processed_sampled_data.to_csv('processed_sampled_data_2.csv')